# Amazon Polly

A refresher on **Amazon Polly** — AWS's managed **text-to-speech (TTS)** service. You send a string of text (or SSML), pick a **voice** and an **engine**, and Polly streams back synthesized audio (MP3/OGG/PCM). It is TTS *only* — AWS's speech-to-text counterpart is a separate service, **Amazon Transcribe**.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the no-key cells (pricing math + SSML builder) run on CPU with stdlib only; the real `SynthesizeSpeech` call is gated behind AWS credentials_

## 1. What & Why

**What it is.** Amazon Polly is a cloud TTS service reached through the AWS SDK (`boto3`'s `polly` client) or the AWS CLI. You hand it text and it returns spoken audio. The headline knobs:

- **Voices** — dozens of named voices across ~40 languages/locales (`Joanna`, `Matthew`, `Amy`, `Lupe`, …), each with a gender and locale.
- **Engines** — four quality/cost tiers: **standard** (concatenative, cheapest), **neural** (NTTS, natural prosody), **long-form** (tuned for paragraphs/articles), and **generative** (most expressive, conversational). Not every voice supports every engine.
- **SSML** — `<speak>…</speak>` markup for pauses, emphasis, pronunciation, plus Amazon extensions (`<amazon:domain name="news">` newscaster style, `<amazon:effect name="whispered">`, `<amazon:breath>`).
- **Speech marks** — request `OutputFormat="json"` to get word/sentence/viseme/ssml timing metadata for lip-sync, captions, or highlighting.
- **Sync vs async** — `synthesize_speech` returns audio inline for short text; `start_speech_synthesis_task` writes long audio (up to 100k billed chars) to **S3** and you poll for completion.

**The problem it solves.** Running your own neural TTS (Coqui, Piper, Tacotron, VITS) means GPUs, model downloads, and vocoder tuning. Polly removes all of that: managed voices, IAM auth, regional endpoints, and pay-per-character billing — in exchange for money and a network round-trip.

**When to reach for it.** Apps already on AWS; IVR/contact-center prompts; accessibility read-aloud; e-learning and audio articles (long-form engine); notifications and voicebots; anything that wants captions/lip-sync via speech marks.

**When not to.** When audio cannot leave your machine (use Piper/Coqui offline); when you need best-in-class expressive cloning of an arbitrary voice (ElevenLabs leads; Polly has no instant voice cloning — its [Brand Voice](https://aws.amazon.com/polly/) is a gated custom engagement); or when you also need STT in the same SDK call (that's Transcribe, a separate service).

## 2. Mental Model

**Polly is a single stateless endpoint: text in → audio bytes out.** There is no session and no model to load on your side. Every call is self-contained — you describe *what to say* (Text or SSML), *who says it* (`VoiceId`), *how well* (`Engine`), and *what container* (`OutputFormat`), and AWS streams the bytes back.

```
            your machine                            AWS (Polly, regional)
 ┌──────────────────────────────────┐  HTTPS   ┌───────────────────────────────┐
 │ boto3.client("polly")            │ ───────▶ │  standard | neural |          │
 │   Text / SSML                    │          │  long-form | generative       │
 │   VoiceId = "Joanna"             │          │  voices x engines             │
 │   Engine  = "neural"             │          │                               │
 │   OutputFormat = "mp3"           │ ◀─────── │  AudioStream  (audio/* bytes) │
 └──────────────────────────────────┘  audio   └───────────────────────────────┘
        │                                              ▲
        │ long text (> sync limit)?                    │  writes .mp3 to your bucket
        └── start_speech_synthesis_task ──────────────▶ │  S3, then poll get_speech_synthesis_task
```

Two axes decide everything:

1. **Quality/cost = `Engine`.** `standard` (cheap, robotic) → `neural` (natural) → `long-form` / `generative` (most expressive, priciest). The voice must support the engine you pick.
2. **Size = sync vs async.** Short text → `synthesize_speech` returns an `AudioStream` you read like a file. Long text → `start_speech_synthesis_task` drops the file in S3 and you poll. The dividing line is the **3,000 billed-character** synchronous cap.

Everything else (SSML, speech marks, lexicons, sample rate) is a parameter on the same one-shot call.

## 3. Key Concepts

- **`VoiceId`.** The named voice, e.g. `Joanna`, `Matthew`, `Ivy` (en-US), `Amy`, `Brian` (en-GB), `Lupe` (es-US), `Mizuki` (ja-JP). Each voice has a fixed gender + locale. List them at runtime with `describe_voices()`, which also reports each voice's **`SupportedEngines`**.
- **`Engine`.** `standard` | `neural` | `long-form` | `generative`. Quality and price climb left→right. **Not every voice supports every engine** — pass the wrong pair and Polly errors. `neural` is the common default; `long-form` suits articles; `generative` is the most lifelike/conversational.
- **`OutputFormat` + `SampleRate`.** `mp3`, `ogg_vorbis`, `pcm` (raw 16-bit signed, for DSP/telephony), or `json` (speech marks, not audio). Sample rates depend on format/engine (e.g. neural: 8000/16000/22050/24000 Hz; pcm has no MP3 header so you must wrap it yourself).
- **`TextType`: `text` vs `ssml`.** Default `text` is plain. `ssml` lets you use `<speak>` markup: `<break>`, `<emphasis>`, `<prosody>`, `<say-as>`, `<phoneme>`, plus Amazon extensions like `<amazon:domain name="news">` (newscaster) and `<amazon:effect name="whispered">`.
- **Billed vs total characters.** Polly bills the **text characters only** — **SSML tags are NOT billed** (unlike Azure, which bills the markup). Tags still count toward the *request size* limit, though. Synchronous `SynthesizeSpeech`: **6,000 total / 3,000 billed** chars. Async `StartSpeechSynthesisTask`: **200,000 total / 100,000 billed** chars.
- **Sync `synthesize_speech` vs async `start_speech_synthesis_task`.** Sync returns an `AudioStream` (a streaming body) inline. Async writes the result to an **S3 bucket** and returns a task id you poll with `get_speech_synthesis_task` until `taskStatus == "completed"`. Use async for anything over the sync cap.
- **Speech marks.** With `OutputFormat="json"` and `SpeechMarkTypes=["word","sentence","viseme","ssml"]`, Polly returns a stream of newline-delimited JSON objects with `time` offsets — the raw material for captions, karaoke-style highlighting, and avatar lip-sync (visemes).
- **Lexicons (PLS).** Upload [PLS](https://www.w3.org/TR/pronunciation-lexicon/) pronunciation lexicons with `put_lexicon`, then pass `LexiconNames=[...]` to fix custom pronunciations (brand names, acronyms) without editing every input string.
- **Pricing (per 1M chars, approx).** standard ≈ \$4, neural ≈ \$16, long-form ≈ \$100, generative ≈ \$30. A 12-month **free tier** covers a monthly allowance per engine. Numbers drift — confirm on the pricing page.

## 4. Setup

```bash
pip install boto3        # the AWS SDK for Python; Polly is boto3.client("polly")

# Credentials (any standard AWS mechanism works): env vars, ~/.aws/credentials, or an IAM role.
export AWS_ACCESS_KEY_ID="<your-key>"
export AWS_SECRET_ACCESS_KEY="<your-secret>"
export AWS_DEFAULT_REGION="us-east-1"      # Polly is regional; pick a region that has the engines you want
```

You need an AWS account and an IAM identity with the `polly:SynthesizeSpeech` (and, for async, `polly:StartSpeechSynthesisTask` + S3 `PutObject`) permissions. No model download and nothing GPU-bound runs locally — all synthesis happens in AWS. The AWS Free Tier includes a Polly allowance for the first 12 months.

The runnable cells below stay **self-contained and credential-free**: Example 1 reproduces the **per-engine pricing math**, Example 2 builds an **SSML document** and shows the billed-vs-total character distinction, and Example 3 makes a **real `synthesize_speech` call** gated behind AWS creds — so the notebook always executes top-to-bottom.

In [ ]:
import sys

print(f"python {sys.version.split()[0]}")
print("Amazon Polly = text in -> audio bytes out. One stateless call: Text/SSML + VoiceId + Engine + OutputFormat.")
print("The next two cells run with NO credentials and NO network: pricing math, then an SSML builder.")

## 5. Worked Examples

### Example 1 — Cost accounting: compare the four engines before you commit

Polly bills **per character of input text** (SSML tags excluded), at very different rates per engine. The quality jump from `standard` → `generative` is real, but so is the 7×+ price jump — and `long-form` is the priciest per character. Before wiring Polly into a product, sanity-check the monthly bill at your expected volume. Pure Python, no credentials. (Rates drift — treat as ballpark and confirm on the pricing page.)

In [ ]:
# Approx. per-1M-character rates + 12-month free-tier monthly allowances (verify on the pricing page).
ENGINES = {
    "standard":   {"per_1m_chars":   4.0, "free_chars": 5_000_000},
    "neural":     {"per_1m_chars":  16.0, "free_chars": 1_000_000},
    "long-form":  {"per_1m_chars": 100.0, "free_chars":   500_000},
    "generative": {"per_1m_chars":  30.0, "free_chars":   100_000},
}

def monthly_cost(chars, engine):
    e = ENGINES[engine]
    billable = max(0, chars - e["free_chars"])
    return billable / 1_000_000 * e["per_1m_chars"]

VOLUME = 3_000_000  # characters synthesized per month
print(f"Volume: {VOLUME:,} billed chars/month\n")
print(f"  {'engine':<12}{'$/1M chars':>12}{'free chars':>14}{'monthly cost':>14}")
for engine, e in ENGINES.items():
    print(f"  {engine:<12}{e['per_1m_chars']:>12.0f}{e['free_chars']:>14,}{monthly_cost(VOLUME, engine):>14.2f}")

cheapest = min(ENGINES, key=lambda k: monthly_cost(VOLUME, k))
priciest = max(ENGINES, key=lambda k: monthly_cost(VOLUME, k))
print(f"\nAt {VOLUME:,} chars/mo: cheapest = {cheapest} (${monthly_cost(VOLUME, cheapest):.2f}), "
      f"priciest = {priciest} (${monthly_cost(VOLUME, priciest):.2f}).")
print("Rule of thumb: prototype on neural; reserve long-form/generative for content where the quality pays for itself.")

### Example 2 — Build the SSML, and see why billed ≠ total characters

The power of Polly beyond one-line TTS is **SSML** — and the key billing wrinkle is that **only the spoken text is billed; the tags are not** (though tags *do* count toward the 6,000-char synchronous request limit). Below we assemble an SSML document with a newscaster domain, a pause, and an emphasis, then strip the tags to compute *billed* characters vs *total* request characters. No network call, so nothing is synthesized and no quota is spent.

In [ ]:
import re

def build_ssml(text, domain="news"):
    # <amazon:domain name="news"> switches a supported neural voice into a newscaster style.
    # (Polly accepts the amazon: prefix without an xmlns declaration; a strict XML parser would not.)
    return (
        "<speak>"
        f"<amazon:domain name='{domain}'>"
        "Breaking news. <break time='400ms'/>"
        f"<emphasis level='strong'>{text}</emphasis> "
        "<say-as interpret-as='date'>2026-06-23</say-as>."
        "</amazon:domain>"
        "</speak>"
    )

ssml = build_ssml("Amazon Polly now speaks in a newscaster style")

# Lightweight indenter: newline + indent on each tag boundary (avoids a strict XML parser).
pretty = re.sub(r"><", ">\n<", ssml)
print("Generated SSML:\n")
print(pretty)

# Billed characters = the document with all <tags> removed (the spoken text only).
billed_text = re.sub(r"<[^>]+>", "", ssml)
total_chars = len(ssml)
billed_chars = len(billed_text)
print(f"\ntotal request characters : {total_chars:>4}  (counts toward the 6,000 sync limit)")
print(f"billed characters        : {billed_chars:>4}  (what AWS actually charges for)")
print(f"tag overhead (free)      : {total_chars - billed_chars:>4}  characters of markup, not billed")
print("\nPass this with TextType='ssml': client.synthesize_speech(Text=ssml, TextType='ssml', ...)")

### Example 3 — Real synthesis with boto3 (gated)

With `boto3` installed and AWS credentials available, synthesis is a single `synthesize_speech` call; the response's `AudioStream` is a streaming body you read like a file and write to disk. This makes a real network call and **consumes quota**, so it's gated behind a credentials check. Either way the cell prints the canonical call shape — text → MP3 — and the `describe_voices` lookup you'd use to confirm a voice supports your engine.

In [ ]:
import os

# Treat presence of an access key OR a configured profile/role as "creds available".
has_creds = bool(os.getenv("AWS_ACCESS_KEY_ID") or os.getenv("AWS_PROFILE"))

if has_creds:
    import boto3

    polly = boto3.client("polly")  # region/creds resolved from env/profile/role

    resp = polly.synthesize_speech(
        Text="Hello from Amazon Polly.",
        VoiceId="Joanna",
        Engine="neural",
        OutputFormat="mp3",
    )
    audio = resp["AudioStream"].read()          # AudioStream is a streaming body
    with open("polly_out.mp3", "wb") as f:
        f.write(audio)
    print(f"Synthesized {len(audio):,} bytes -> polly_out.mp3 ({resp['ContentType']})")

    # Which engines does this voice support?
    voices = polly.describe_voices(LanguageCode="en-US")["Voices"]
    joanna = next(v for v in voices if v["Id"] == "Joanna")
    print(f"Voice Joanna supports engines: {joanna['SupportedEngines']}")
else:
    print("Set AWS credentials (AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY, or AWS_PROFILE) and "
          "`pip install boto3` to run for real.\n")
    print("import boto3")
    print("polly = boto3.client('polly')")
    print("resp = polly.synthesize_speech(")
    print("    Text='Hello from Amazon Polly.', VoiceId='Joanna',")
    print("    Engine='neural', OutputFormat='mp3')")
    print("open('polly_out.mp3', 'wb').write(resp['AudioStream'].read())\n")
    print("# Long text (> 3,000 billed chars) -> async to S3:")
    print("task = polly.start_speech_synthesis_task(")
    print("    Text=long_text, VoiceId='Joanna', Engine='neural',")
    print("    OutputFormat='mp3', OutputS3BucketName='my-bucket')")
    print("# then poll polly.get_speech_synthesis_task(TaskId=task['SynthesisTask']['TaskId'])")

## 6. Gotchas & Pitfalls

- **Voice ⇄ engine compatibility is not universal.** Not every `VoiceId` supports every `Engine`. `generative` and `long-form` are available on a *subset* of voices; pass an unsupported pair and you get a `ValidationException`. Call `describe_voices()` and check `SupportedEngines` instead of assuming.
- **The 3,000 billed-character sync cap is silent until you hit it.** `synthesize_speech` rejects requests over **6,000 total / 3,000 billed** characters with a `TextLengthExceededException`. Anything longer (articles, chapters) **must** go through `start_speech_synthesis_task` → S3. Don't discover this in production.
- **SSML tags are free but still count toward the size limit.** Unlike Azure, Polly does **not** bill the markup — but heavy tagging can still push your *total* characters past 6,000 on a sync call even when *billed* characters are well under 3,000. Budget the request by total chars, the bill by billed chars.
- **`AudioStream` is a stream you must read once.** The response value is a streaming body, not bytes. Call `.read()` (and ideally close it / use the client in a context); reading it twice yields empty data, and forgetting to read it leaks the connection.
- **PCM output has no container.** `OutputFormat="pcm"` returns raw signed 16-bit little-endian samples with **no WAV header**. If you write it straight to `.wav` it won't play — wrap it (e.g. Python's `wave` module) or use `mp3`/`ogg_vorbis` which are self-describing.
- **Region matters for both latency and feature availability.** Polly is regional; newer engines/voices roll out to some regions before others. A voice/engine that works in `us-east-1` may not yet exist in your region — and cross-region calls add latency.
- **Async output overwrites by key, and tasks expire.** `start_speech_synthesis_task` names the S3 object by task id under your prefix; the task record (and its `OutputUri`) is only retained for a limited window, so fetch the result reasonably promptly rather than assuming it persists indefinitely.
- **Lexicons are regional and must be attached per call.** A `put_lexicon` lexicon lives in one region and only applies when you pass it via `LexiconNames=[...]` — uploading it doesn't make it global or automatic.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs Amazon Polly |
|---|---|---|
| **Amazon Polly** | AWS-native apps, IVR/notifications, long-form articles, captions/lip-sync via speech marks, pay-per-char | Cloud-only; TTS only (no STT); top expressive quality trails ElevenLabs; no instant voice cloning |
| **Amazon Transcribe** | The STT half of AWS (Polly is TTS only) | Different service entirely — pair them; not a Polly substitute |
| **Azure AI Speech** | Azure-native, one SDK for STT+TTS+translation, emotional styles, pronunciation assessment | Tied to Azure; bills SSML markup; comparable quality — choose by cloud |
| **Google Cloud TTS** | GCP-native, broad voice catalog, telephony/Studio voices | Tied to GCP; comparable quality and pricing model |
| **ElevenLabs** | Best-in-class naturalness, instant voice cloning, emotive TTS | Pricier at quality tier; not cloud-native to AWS; weaker enterprise/IAM plumbing |
| **OpenAI TTS** | Simple, cheap, decent quality inside the OpenAI stack | Fewer voices, limited SSML, less enterprise plumbing |
| **Piper / Coqui / VITS (self-hosted)** | Offline/on-device, no per-use bill, data never leaves the box | You run the ops/GPUs; no managed scaling; quality trails hosted neural/generative |

**Rule of thumb:** choose **Amazon Polly when you're on AWS and want managed, pay-per-character TTS with IAM auth, S3 async for long content, and speech marks for captions/lip-sync.** Reach for **ElevenLabs** when expressive quality or voice cloning is the deciding factor, **Azure/Google** to stay native in those clouds (and for a unified STT+TTS SDK on Azure), **OpenAI** for a cheap simple stack, and **Piper/Coqui/VITS** when audio must stay offline or you're optimizing high-volume cost. For STT on AWS, reach for **Amazon Transcribe**, not Polly.

## 8. Resources

- **Amazon Polly Developer Guide** — concepts, voices, engines, SSML, async: https://docs.aws.amazon.com/polly/latest/dg/what-is.html
- **Boto3 Polly client reference** — every API call and parameter: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/polly.html
- **Supported SSML tags (incl. Amazon extensions)** — `<amazon:domain>`, `<amazon:effect>`, `<break>`, `<phoneme>`: https://docs.aws.amazon.com/polly/latest/dg/supportedtags.html
- **Voices & languages (per-engine support)** — the full voice catalog: https://docs.aws.amazon.com/polly/latest/dg/voicelist.html
- **Speech marks** — word/sentence/viseme/ssml timing metadata: https://docs.aws.amazon.com/polly/latest/dg/speechmarks.html
- **Long-form / async synthesis to S3** — `StartSpeechSynthesisTask`: https://docs.aws.amazon.com/polly/latest/dg/asynchronous.html
- **Managing lexicons (PLS)** — custom pronunciations: https://docs.aws.amazon.com/polly/latest/dg/managing-lexicons.html
- **Pricing** — per-engine rates, free tier: https://aws.amazon.com/polly/pricing/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
ENGINES = {}
SYNC_TOTAL, SYNC_BILLED = 6_000, 3_000


def billed_characters(text, text_type="text"):
    ...


def plan_request(text, voice, engine, voices, text_type="text"):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE